### 10 - Building files to test dependency parsing
In this notebook, we will be build a function which can parse sentences with certain attributes (such as not containing unparsed readings, containing a locative noun, etc.) to more specifically test our dependency grammar.

In [15]:
import csv
import ast
import string
import rich
from pathlib import Path
from typing import Tuple, Iterable

# Helper functions
# skip punctuation readings
def is_punct(form: str) -> bool:
    return all(c in string.punctuation for c in form)

# check that at least one analysis has every tag in tags
def token_has_tags(analyses: list[str], tags: Iterable[str]) -> bool:
    return all(any(tag in ana for ana in analyses) for tag in tags)

def extract_examples(
    csv_path: str | Path,
    *,
    limit: int = 50,
    ojibwe_out: str | Path,
    english_out: str | Path,
    using_patterns: bool = False,
    patterns: tuple[str, ...] = None,
    require_ambiguity: bool = False,
    ambiguous_sets: tuple[tuple[str, ...], ...] | None = None,
) -> None:
    """
    Read csv file formatted with Ojibwe, English, and FST readings,
    find rows whose fst_readings contains no missing analyses and 
    at least one analysis with a tag from PATTERNS (if set), and
    write up to 'limit' matching sentences into two parallel files.

    Each output file has one sentence per line, corresponding lines align.
    """
    picked = 0
    seen = set()        # avoid duplicates

    with (
        open(csv_path, newline="", encoding="utf-8") as fin,
        open(ojibwe_out,  "w", encoding="utf-8") as f_oj,
        open(english_out, "w", encoding="utf-8") as f_en,
    ):
        reader = csv.DictReader(fin)

        for row in reader:
            if picked >= limit:
                break

            # 1. Recover analyses list
            try:
                analyses = ast.literal_eval(row["fst_readings"])
            except Exception:
                continue                     

            # 2. Skip rows with unparsed tokens excluding punctuation
            if any(
                (not tok.get("fst_analyses"))
                and (not is_punct(tok.get("word_form", "")))
                for tok in analyses
            ):
                continue
            
            # 3. Require at least one tag in patterns (if set)
            if using_patterns: 
                if not patterns:
                    return rich.print(f"[bold red] Error: patterns empty but using_patterns=True")
                
                pattern_found = any(
                    any(pat in ana.split("+") for pat in patterns)
                    for tok in analyses
                    for ana in tok.get("fst_analyses", [])
                )               
    
                if not pattern_found:
                   continue

            # 4. Require readings to be ambiguous (if set)   
            if require_ambiguity:
                if not ambiguous_sets:
                    rich.print("[bold red] Error: ambiguous_sets empty but require_ambiguity=True")
                    return
                ambiguous_token_found = any(
                    any(
                        token_has_tags(tok.get("fst_analyses", []), tag_group)
                        and len(tok.get("fst_analyses", [])) >= 2
                        for tag_group in ambiguous_sets
                    )
                    for tok in analyses
                )
                if not ambiguous_token_found:
                    continue

            # 5. Deduplicate sentence pairs
            oj_sent, en_sent = row["ojibwe"].strip(), row["english"].strip()
            if (oj_sent, en_sent) in seen:
                continue
            seen.add((oj_sent, en_sent))

            # 6. Write to files
            f_oj.write(oj_sent + "\n")
            f_en.write(en_sent + "\n")
            picked += 1

<H4>Example usage:

Set paths:

In [18]:
AMBIGUOUS_SENTS_PATH = "../data/parallel_data/raw/sentences_with_ambiguity.csv"
OJIBWE_OUT_PATH = "../data/parallel_data/treebank_sentences/ojibwe_mantmpdeg.txt"
ENGLISH_OUT_PATH = "../data/parallel_data/treebank_sentences/english_mantmpdeg.txt"

Call function:

In [ ]:
# modify or add new tuples to check for different patterns
LOC_PATTERNS = (
    "ADVLoc",        # adverb locative tag
    "+Loc",          # locative suffix in FST string
    " Loc",          # space-locative
)


extract_examples(csv_path=AMBIGUOUS_SENTS_PATH, ojibwe_out=OJIBWE_OUT_PATH, english_out=ENGLISH_OUT_PATH, using_patterns=False)


In [ ]:
NA_NI = ("NA",  "NI")
NAD_NID = ("NAD", "NID") 
extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    require_ambiguity=True,
    ambiguous_sets=(NA_NI, NAD_NID),  
)

In [14]:
ZERO_OBV = ("0SgObvSubj", "0SgObvObj", "0PlObvSubj", "0PlObvObj", "0SgObV", "0PlObv")   
extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    using_patterns=True,
    patterns=ZERO_OBV 
)

In [4]:
VAI_VII = ("VAI",  "VII")
VTA_VTI = ("VTA", "VTI")
VII_VTI = ("VII", "VTI") 
VAI_VTI = ("VAI", "VTI")
extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    require_ambiguity=True,
    ambiguous_sets=(VAI_VII, VTA_VTI, VII_VTI, VAI_VTI),  
)

In [13]:
VTI = ("VTI",)
extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    using_patterns=True,
    patterns=VTI   
)

In [17]:
SPECIAL_PRONOUNS = ("PRONInter", "PRONIndf", "PRONPret",)
extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    using_patterns=True,
    patterns=SPECIAL_PRONOUNS,  
)

In [19]:
MAN_DEG_TMP = ("ADVMan", "ADVDeg", "ADVTmp",)
extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    using_patterns=True,
    patterns=MAN_DEG_TMP,  
)

In [20]:
AMBIGUOUS_SENTS_PATH = "../data/parallel_data/raw/sentences_with_ambiguity.csv"
OJIBWE_OUT_PATH = "../data/parallel_data/treebank_sentences/ojibwe_num.txt"
ENGLISH_OUT_PATH = "../data/parallel_data/treebank_sentences/english_num.txt"

NUM = ("NUM",)
extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    using_patterns=True,
    patterns=NUM,  
)

In [ ]:
AMBIGUOUS_SENTS_PATH = "../data/parallel_data/raw/sentences_with_ambiguity.csv"
OJIBWE_OUT_PATH = "../data/parallel_data/treebank_sentences/ojibwe_disambiguation_100.txt"
ENGLISH_OUT_PATH = "../data/parallel_data/treebank_sentences/english_disambiguation_100.txt"

extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
)

In [18]:
from itertools import product 

AMBIGUOUS_SENTS_PATH = "../data/parallel_data/raw/sentences_with_ambiguity.csv"
OJIBWE_OUT_PATH = "../data/parallel_data/treebank_sentences/ojibwe_V_ADV.txt"
ENGLISH_OUT_PATH = "../data/parallel_data/treebank_sentences/english_V_ADV.txt"

VERB = ("VTA", "VAI", "VTI", "VII", "VAIO")
ADV  = (
    "ADVConj", "ADVDisc", "ADVDub", "ADVGram",
    "ADVInter", "ADVLoc", "ADVMan", "ADVNeg",
    "ADVPred", "ADVQnt", "ADVTmp", "AVDDeg"
)

verb_adv_pairs = tuple(product(VERB, ADV))

extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    require_ambiguity=True,
    ambiguous_sets=verb_adv_pairs,
)




In [20]:
from itertools import product 

AMBIGUOUS_SENTS_PATH = "../data/parallel_data/raw/sentences_with_ambiguity.csv"
OJIBWE_OUT_PATH = "../data/parallel_data/treebank_sentences/ojibwe_N_ADV.txt"
ENGLISH_OUT_PATH = "../data/parallel_data/treebank_sentences/english_N_ADV.txt"

NOUN = ("NA", "NI", "NAD", "NID")
ADV  = (
    "ADVConj", "ADVDisc", "ADVDub", "ADVGram",
    "ADVInter", "ADVLoc", "ADVMan", "ADVNeg",
    "ADVPred", "ADVQnt", "ADVTmp", "AVDDeg"
)

noun_adv_pairs = tuple(product(NOUN, ADV))

extract_examples(
    csv_path=AMBIGUOUS_SENTS_PATH,
    ojibwe_out=OJIBWE_OUT_PATH,
    english_out=ENGLISH_OUT_PATH,
    require_ambiguity=True,
    ambiguous_sets=noun_adv_pairs,
)
